# English - feature extraction


The script below applied the [`Marqo/dunzhang-stella_en_400M_v5`](https://huggingface.co/Marqo/dunzhang-stella_en_400M_v5) text-analysis model. This is an updated version of [`dunzhang/stella_en_400M_v5`](https://huggingface.co/NovaSearch/stella_en_400M_v5), containing a fused Matryoshka Layer which reduces the computational overhead for generating embeddings, while leaving relevance metrics unchanged. The design of [`dunzhang/stella_en_400M_v5`](https://huggingface.co/NovaSearch/stella_en_400M_v5) produces embeddings at multiple dimensionalities (512 up to 8192), with 1024-dimensional vectors offering near-optimal performance at lower computational cost. The model is approximately 400 million parameters as indicated by the "400M" in its name. Stella is optimized for semantic retrieval and similarity tasks, supports instruction-style prompting (e.g. query-to-passage vs. sentence-to-sentence), and is trained with a recommended maximum sequence length of 512 tokens. While it achieves excellent benchmark results (e.g. MTEB), it is not suited for long-document encoding: text longer than ~512 tokens must be truncated, leading to information loss. Compared to long-context models such as BGE-M3, Stella prioritizes retrieval quality and efficiency on short-to-medium texts over full long-text coverage.

The size on disk would be roughly 1.5-2 GB, depending on the exact storage format. The model requires a GPU to run more effectively. A regular laptop might not have enough computational power.

Here's a more extended [**list**](https://huggingface.co/models?library=sentence-transformers&sort=downloads) with multiple transformer models in Huggin Face.

In [2]:
# import libraries
import os
import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig # for english embeddings
from sklearn.preprocessing import normalize # for english embeddings
from FlagEmbedding import BGEM3FlagModel # for dutch embeddings
import pandas as pd
import numpy as np
from tqdm import tqdm

# check whether GPU or CPU is available for pytorch
print(torch.cuda.is_available()) # Return True if GPU is available, otherwise False
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

c:\Users\aks284\AppData\Local\miniconda3\envs\gitp_llm_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False
CPU


In [ ]:
# Load the transformer embeddings
model_dir = "Marqo/dunzhang-stella_en_400M_v5"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(
    model_dir, trust_remote_code=True
).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)

c:\Users\aks284\AppData\Local\miniconda3\envs\gitp_llm_project\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aks284\.cache\huggingface\hub\models--Marqo--dunzhang-stella_en_400M_v5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
A new version of the following files was downloaded from https://hugging

In [35]:
# Load the data
df = pd.read_csv("input/data_gitp_short.csv")

In [44]:
# df.columns

In [33]:
# Function to compute mean-pooled embeddings
def get_mean_embedding(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(model.device)
    
    with torch.inference_mode():
        outputs = model(**inputs)
    
    last_hidden_state = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    
    masked_embeddings = last_hidden_state * attention_mask
    summed = masked_embeddings.sum(dim=1)
    counts = attention_mask.sum(dim=1)
    mean_pooled = summed / counts
    
    return mean_pooled.cpu().numpy()

In [39]:
# ===== MANUAL CONFIGURATION =====
# Set these 3 variables as needed:
target_column = 'Beoordeling.all_en'  # Change to your column
embedding_prefix = 'en_'              # Change prefix for column names
text_columns = [target_column]           # Keep as list

# Create fresh dataframe with only needed columns
df_to_process = df[['id', target_column]].copy()

In [41]:
df_to_process.head()

,id,Beoordeling.all_en
0,1460133,"Oh yeah, Carina, I can imagine you're facing a..."
1,1462073,"Marine, yeah, I do understand that you’re busy..."
2,1549902,"Did you think countries, yeah, nice that you c..."
3,1552130,"Hi Marine, nice to work with you. Like you sai..."
4,1569467,"How nice that we’re talking to each other now,..."


In [42]:
# ===== ENCODING =====
for col in text_columns:
    print(f"Encoding column: {col}")
    texts = df_to_process[col].fillna("").tolist()
    all_embeddings = []
    
    batch_size = 8
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_embeddings = get_mean_embedding(batch)
        all_embeddings.extend(batch_embeddings)
        torch.cuda.empty_cache()
    
    # Add ALL embedding dimensions at once (no PerformanceWarning)
    emb_dim = len(all_embeddings[0])
    embedding_data = {f"{embedding_prefix}{dim}": [vec[dim] for vec in all_embeddings] 
                      for dim in range(emb_dim)}
    df_embeddings = pd.DataFrame(embedding_data)
    df_to_process = pd.concat([df_to_process, df_embeddings], axis=1)

Encoding column: Beoordeling.all_en


  0%|          | 0/109 [00:00<?, ?it/s]c:\Users\aks284\AppData\Local\miniconda3\envs\gitp_llm_project\Lib\site-packages\transformers\modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
100%|██████████| 109/109 [1:02:36<00:00, 34.46s/it]


In [43]:
# ===== SAVING =====
csv_path = f"embeddings/embeddings_{embedding_prefix.rstrip('_')}.csv"
excel_path = f"embeddings/embeddings_{embedding_prefix.rstrip('_')}.xlsx"

df_to_process.to_csv(csv_path, index=False)
df_to_process.to_excel(excel_path, index=False)
print(f"Saved embeddings to {csv_path} and {excel_path}")
print(f"Created {emb_dim} columns: {embedding_prefix}0 to {embedding_prefix}{emb_dim-1}")

Saved embeddings to embeddings/embeddings_en.csv and embeddings/embeddings_en.xlsx
Created 1024 columns: en_0 to en_1023


# Dutch - feature extraction

## Option A: [`BAAI/bge-m3`](https://huggingface.co/BAAI/bge-m3) pre-trained multilingual embedding model.

BGE-M3 is a large, versatile embedding model designed for multi-functionality, multi-linguality, and multi-granularity. It supports more than 100 languages, including Dutch, and can process text ranging from short sentences to long documents of up to 8,192 tokens, making it well suited for participant-level texts of several hundred words.

The model produces 1024-dimensional dense embeddings and is jointly trained to support dense retrieval, sparse (lexical) retrieval, and multi-vector (ColBERT-style) retrieval within a single architecture. For standard embedding use cases, it can be applied as a bi-encoder in the same way as other BGE models, without requiring instruction prompts.

Due to its size and long-context capability, BGE-M3 is computationally heavier than lightweight sentence-transformer models and benefits strongly from GPU acceleration, although CPU execution is possible at higher latency. Compared to compact multilingual models, BGE-M3 prioritizes coverage of long documents and representational richness over speed and simplicity.

The model is approximately 2.2GB.

## Option B: [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) pre-trained sentence-transformer model.

This is a lightweight, multilingual embedding model trained using the Sentence-BERT (SBERT) framework. It supports over 50 languages, including Dutch, and is well suited for semantic similarity, clustering, and supervised downstream tasks.

The model maps sentences and paragraphs into a 384-dimensional dense vector space using a MiniLM-based transformer encoder with mean pooling over token embeddings. It has a maximum sequence length of 128 tokens, making it efficient but less suitable for very long documents without chunking.

With approximately 120 million parameters, the model is fast, memory-efficient, and can comfortably run on CPU-only machines, while still benefiting from GPU acceleration if available. The on-disk size is roughly 400–500 MB, depending on format.

Compared to large English-only models (e.g., Stella), this model trades off raw representational power for multilingual coverage, speed, and ease of deployment, making it a safe and widely used default for Dutch text embeddings.

One limitation of this model is that it processes up to 128 tokens (approximately 100 words). This means that a large part of the verbal information (~700 words per participant) won't be captured. Additionally, it outputs 384 vectors, substantially fewer than the 1024 vectors of the english model, something that creates an unequal comparison.

In [3]:
# Load the BGE-M3 embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=(device == "cuda")
)

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 91980.35it/s]


In [4]:
# Function to compute dense embeddings using BGE-M3
def get_dense_embeddings(texts, batch_size=8, max_length=8192):
    outputs = model.encode(
        texts,
        batch_size=batch_size,
        max_length=max_length,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    return outputs["dense_vecs"]


In [10]:
# Load the data
df = pd.read_csv("input/data_gitp_short.csv")

In [11]:
# ===== MANUAL CONFIGURATION =====
target_column = 'Beoordeling.all_en_tw'
embedding_prefix = 'en_tw_'
text_columns = [target_column]

df_to_process = df[['id', target_column]].copy()

In [12]:
df_to_process.head()

,id,Beoordeling.all_en_tw
0,1460133,"Oh yeah, PERSON_FIRSTNAME_1, I can imagine you..."
1,1462073,"Marine, yeah, I do understand that you’re busy..."
2,1549902,"Did you think countries, yeah, nice that you c..."
3,1552130,"Hi PERSON_FIRSTNAME_1, nice to work with you. ..."
4,1569467,"How nice that we’re talking to each other now,..."


In [13]:
# ===== ENCODING =====
for col in text_columns:
    print(f"Encoding column: {col}")
    texts = df_to_process[col].fillna("").tolist()

    all_embeddings = get_dense_embeddings(
        texts,
        batch_size=8,
        max_length=8192
    )

    emb_dim = all_embeddings.shape[1]

    embedding_data = {
        f"{embedding_prefix}{dim}": all_embeddings[:, dim]
        for dim in range(emb_dim)
    }

    df_embeddings = pd.DataFrame(embedding_data)
    df_to_process = pd.concat([df_to_process, df_embeddings], axis=1)


Encoding column: Beoordeling.all_en_tw


pre tokenize:   0%|          | 0/109 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 109/109 [51:29<00:00, 28.34s/it] 


In [14]:
# ===== SAVING =====
csv_path = f"embeddings/embeddings_{embedding_prefix.rstrip('_')}.csv"
excel_path = f"embeddings/embeddings_{embedding_prefix.rstrip('_')}.xlsx"

df_to_process.to_csv(csv_path, index=False)
df_to_process.to_excel(excel_path, index=False)

print(f"Saved embeddings to {csv_path} and {excel_path}")
print(f"Created {emb_dim} columns: {embedding_prefix}0 to {embedding_prefix}{emb_dim-1}")


Saved embeddings to embeddings/embeddings_en_tw.csv and embeddings/embeddings_en_tw.xlsx
Created 1024 columns: en_tw_0 to en_tw_1023
